# Module 22 — Production Agent Architecture

> **SDKs:** `sqlite3`, `dataclasses`

| Part | Topic |
|------|-------|
| **1** | State & Persistence — Checkpointing workflows |
| **2** | Scale & Resilience — Queue-based execution |


---
## Part 1 & 2 — Persistence and Resilient Queues

In production, agents crash. Workflows must be persisted in databases to survive pod evictions and resume gracefully.

In [1]:
import sqlite3, json

class ProductionOrchestrator:
    def __init__(self):
        self.conn = sqlite3.connect(":memory:")
        self.conn.execute("CREATE TABLE jobs (id TEXT, status TEXT, state TEXT)")
        
    def enqueue(self, job_id: str, state: dict):
        self.conn.execute("INSERT INTO jobs VALUES (?, 'PENDING', ?)", (job_id, json.dumps(state)))
        print(f"  [Queue] Job {job_id} enqueued.")
        
    def process_next(self):
        cur = self.conn.execute("SELECT id, state FROM jobs WHERE status='PENDING' LIMIT 1")
        row = cur.fetchone()
        if not row: return
        
        job_id, state_json = row
        state = json.loads(state_json)
        print(f"  [Worker] Processing {job_id}... State: {state}")
        
        # Simulate work
        state["step"] = "COMPLETED"
        
        self.conn.execute("UPDATE jobs SET status='DONE', state=? WHERE id=?", (json.dumps(state), job_id))
        print(f"  [Worker] Job {job_id} finished and state persisted.")

orch = ProductionOrchestrator()
print("🏗️  Production Architecture Demo")
print("=" * 60)
orch.enqueue("task-881", {"query": "Summarize logs", "step": "INIT"})
orch.process_next()


🏗️  Production Architecture Demo
  [Queue] Job task-881 enqueued.
  [Worker] Processing task-881... State: {'query': 'Summarize logs', 'step': 'INIT'}
  [Worker] Job task-881 finished and state persisted.
